In [44]:
import pandas as pd
import numpy as np
import os
import re


In [45]:
print("DATA PREPROCESSING - CONSUMER COMPLAINTS")
print("\nSTEP 1: Loading CSV file...")

csv_path = '../data/raw_data/consumer_complaints.csv'

try:
    df = pd.read_csv(csv_path)
    print(f"✓ Successfully loaded!")
    print(f"  Shape: {df.shape}")
except FileNotFoundError:
    print(f"❌ File not found: {csv_path}")
    print("Please check the file path!")
    exit()


DATA PREPROCESSING - CONSUMER COMPLAINTS

STEP 1: Loading CSV file...


/var/folders/rc/z6_750794cgbrljxmmzvgzfw0000gn/T/ipykernel_82881/1214194040.py:7: DtypeWarning: Columns (0: Sub-issue, 1: Consumer complaint narrative, 2: Company public response, 3: Consumer consent provided?, 4: Consumer disputed?) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


✓ Successfully loaded!
  Shape: (1282355, 18)


In [46]:
print("STEP 2: Exploring data structure with analysis...")

print(f"\nTotal records: {len(df):,}")
print(f"Total columns: {len(df.columns)}")

print("\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")

print("\nData types:")
print(df.dtypes)

print("\nMissing value analysis (%):")
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0].round(2).to_string())

print("\nDuplicate rows:")
print(f"  {df.duplicated().sum():,}")

print("\nFirst 3 rows:")
print(df.head(3))


STEP 2: Exploring data structure with analysis...

Total records: 1,282,355
Total columns: 18

Column names:
  1. Date received
  2. Product
  3. Sub-product
  4. Issue
  5. Sub-issue
  6. Consumer complaint narrative
  7. Company public response
  8. Company
  9. State
  10. ZIP code
  11. Tags
  12. Consumer consent provided?
  13. Submitted via
  14. Date sent to company
  15. Company response to consumer
  16. Timely response?
  17. Consumer disputed?
  18. Complaint ID

Data types:
Date received                     str
Product                           str
Sub-product                       str
Issue                             str
Sub-issue                         str
Consumer complaint narrative      str
Company public response           str
Company                           str
State                             str
ZIP code                          str
Tags                              str
Consumer consent provided?        str
Submitted via                     str
Date sent to c

In [47]:
print("STEP 3: Selecting model columns...")

feature_cols = ["Issue", "Sub-issue"]
target_col = "Product"
required_cols = feature_cols + [target_col]

missing_required = [col for col in required_cols if col not in df.columns]
if missing_required:
    print("\nERROR: Required columns are missing from dataset:")
    for col in missing_required:
        print(f"  - {col}")
    exit()

print("\nUsing columns:")
print(f"  Features: {feature_cols}")
print(f"  Target: {target_col}")


STEP 3: Selecting model columns...

Using columns:
  Features: ['Issue', 'Sub-issue']
  Target: Product


In [48]:
print("STEP 4: Extracting columns and handling missing values...")

df_clean = df[required_cols].copy()
print(f"\nRows before cleaning: {len(df_clean):,}")

print("\nMissing values before handling:")
for col in required_cols:
    print(f"  {col}: {df_clean[col].isna().sum():,}")

# Target must exist; remove rows without target
df_clean = df_clean.dropna(subset=[target_col])

# Keep rows with at least one feature available
df_clean = df_clean.dropna(subset=feature_cols, how="all")

# Fill remaining feature NaNs to support consistent text cleaning
for col in feature_cols:
    df_clean[col] = df_clean[col].fillna("")

print(f"\nRows after missing value handling: {len(df_clean):,}")


STEP 4: Extracting columns and handling missing values...

Rows before cleaning: 1,282,355

Missing values before handling:
  Issue: 0
  Sub-issue: 531,186
  Product: 0

Rows after missing value handling: 1,282,355


In [49]:
print("STEP 5: Analyzing unique categories...")

for col in required_cols:
    print(f"\nColumn: {col}")
    print(f"  Unique values: {df_clean[col].nunique():,}")
    print("  Top 10 values:")
    print(df_clean[col].value_counts().head(10))


STEP 5: Analyzing unique categories...

Column: Issue
  Unique values: 167
  Top 10 values:
Issue
Incorrect information on your report                                                134338
Loan modification,collection,foreclosure                                            112311
Incorrect information on credit report                                              102686
Loan servicing, payments, escrow account                                             77333
Cont'd attempts collect debt not owed                                                60687
Problem with a credit reporting company's investigation into an existing problem     51300
Attempts to collect debt not owed                                                    43021
Account opening, closing, or management                                              37961
Communication tactics                                                                35403
Improper use of your report                                                        

In [50]:
print("STEP 6: Cleaning feature text columns...")


def clean_text(text):
    text = str(text)
    if text.lower() in {"nan", "none", "null", "na", "n/a"}:
        return ""

    text = text.lower()
    text = re.sub(r"https?://\\S+|www\\.\\S+", " ", text)  # remove URLs
    text = text.replace("/", " ").replace("-", " ")
    text = re.sub(r"[^a-z0-9\\s]", " ", text)  # remove special symbols
    text = re.sub(r"\\s+", " ", text).strip()  # normalize whitespace
    return text


for col in feature_cols:
    df_clean[col] = df_clean[col].map(clean_text)

# Remove rows where both cleaned feature fields are empty
feature_non_empty_mask = (df_clean[feature_cols[0]].str.len() > 0) | (df_clean[feature_cols[1]].str.len() > 0)
df_clean = df_clean[feature_non_empty_mask].copy()

# Clean target text formatting
# (keeps original label semantics while normalizing spaces)
df_clean[target_col] = df_clean[target_col].astype(str).str.strip().str.replace(r"\\s+", " ", regex=True)

print(f"Rows after text cleaning: {len(df_clean):,}")


STEP 6: Cleaning feature text columns...
Rows after text cleaning: 1,282,355


In [51]:
print("STEP 7: Final verification...")

print(f"\nFinal dataset shape: {df_clean.shape}")
print("\nFinal missing values:")
print(df_clean.isna().sum())

print("\nSample cleaned data:")
print(df_clean.head(5))


STEP 7: Final verification...

Final dataset shape: (1282355, 3)

Final missing values:
Issue        0
Sub-issue    0
Product      0
dtype: int64

Sample cleaned data:
                                  Issue  \
0                   managing an account   
1                   managing an account   
2                 communication tactics   
3  incorrect information on your report   
4                   managing an account   

                                      Sub-issue  \
0             problem using a debit or atm card   
1                      deposits and withdrawals   
2                    frequent or repeated calls   
3  old information reappears or never goes away   
4                                banking errors   

                                             Product  
0                        Checking or savings account  
1                        Checking or savings account  
2                                    Debt collection  
3  Credit reporting, credit repair services, o

In [52]:
print("STEP 8: Saving cleaned data...")

# Notebook-local output path
os.makedirs('../data', exist_ok=True)
output_path_local = '../data/complaints_cleaned.csv'
df_clean.to_csv(output_path_local, index=False)
print(f"✓ Cleaned data saved to: {output_path_local}")

# Project-level output path (mentioned path)
os.makedirs('../data/preprocess_data', exist_ok=True)
output_path_project = '../data/preprocess_data/complaints_cleaned.csv'
df_clean.to_csv(output_path_project, index=False)
print(f"✓ Cleaned data saved to: {output_path_project}")


STEP 8: Saving cleaned data...
✓ Cleaned data saved to: ../data/complaints_cleaned.csv
✓ Cleaned data saved to: ../data/preprocess_data/complaints_cleaned.csv


In [53]:
print("PREPROCESSING COMPLETE!")
print(f"""
Summary:
--------
✓ Loaded: {csv_path}
✓ Features used: {feature_cols}
✓ Target used: {target_col}
✓ Records processed: {len(df_clean):,}
✓ Categories in target: {df_clean[target_col].nunique():,}
✓ Output files:
  - {output_path_local}
  - {output_path_project}

Ready for training!
""")


PREPROCESSING COMPLETE!

Summary:
--------
✓ Loaded: ../data/raw_data/consumer_complaints.csv
✓ Features used: ['Issue', 'Sub-issue']
✓ Target used: Product
✓ Records processed: 1,282,355
✓ Categories in target: 18
✓ Output files:
  - ../data/complaints_cleaned.csv
  - ../data/preprocess_data/complaints_cleaned.csv

Ready for training!

